# Gëstukaay : test de la voix en wolof sur GPU NVIDIA

Ce notebook télécharge **nos deux serveurs de voix depuis le dépôt** (`transcription/serveur.py` et `synthese/serveur.py`, branche `main`) et les fait tourner, avec les versions épinglées des images Docker (torch 2.11 avec CUDA 12.6). Il mesure :
- que les deux modèles se chargent sur le GPU (CUDA) ;
- la VRAM utilisée par les deux ensemble ;
- le temps d'une réponse vocale en wolof (Oolel) et celui d'une transcription (M-Kiriku), sur l'aller-retour.

**Avant de lancer** (Kaggle) :
1. *Settings* → *Accelerator* : **GPU T4 x2** (ou P100) ; *Internet* : **on**.
2. *Add-ons* → *Secrets* : ajouter `HF_TOKEN`, un jeton de lecture Hugging Face, après avoir accepté les conditions de https://huggingface.co/AIHubSN/M-Kiriku-ASR.
3. *Run all*. Durée : environ 15 minutes (installation et téléchargement d'environ 15 Go de modèles).

In [ ]:
# Place disque : /kaggle/working est limité (~20 Go) ; environnements et modèles vont dans /tmp
!df -h /kaggle/working /tmp | cat

In [ ]:
# Deux environnements Python 3.12, exactement comme les images Docker (Kaggle est en 3.13)
!pip install -q uv
import os
os.environ["UV_HTTP_TIMEOUT"] = "300"
!uv venv -q -p 3.12 /tmp/venv_tts && uv pip install -q --no-cache --python /tmp/venv_tts/bin/python \
    --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu126 \
    "torch==2.11.0+cu126" "torchaudio==2.11.0+cu126" "transformers==4.46.3" "diffusers==0.29.0" \
    "conformer==0.3.2" "librosa==1.0.0" "s3tokenizer==0.3.0" "safetensors==0.8.0" "huggingface_hub==0.36.2" \
    "omegaconf==2.3.1" "einops==0.8.2" "scipy==1.18.1" "numpy==2.5.3" "fastapi==0.142.2" "uvicorn==0.54.0"
!uv venv -q -p 3.12 /tmp/venv_asr && uv pip install -q --no-cache --python /tmp/venv_asr/bin/python \
    --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu126 \
    "torch==2.11.0+cu126" "transformers==4.57.6" "av==19.0.1" "numpy==2.5.3" \
    "fastapi==0.142.2" "uvicorn==0.54.0" "python-multipart==0.0.32"
!/tmp/venv_tts/bin/python -c "import torch; print('tts', torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!/tmp/venv_asr/bin/python -c "import torch; print('asr', torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!df -h /tmp | cat

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HOME"] = "/tmp/modeles"  # ~15 Go : hors de /kaggle/working (limité à ~20 Go)
os.environ["TRANSCRIPTION_CLE"] = os.environ["SYNTHESE_CLE"] = "essai-kaggle"
print("jeton chargé (non affiché)")

In [ ]:
# Les deux serveurs tels qu'ils sont dans le dépôt (main) : c'est bien le code déployé qui est mesuré
!curl -fsSL -o serveur_transcription.py https://raw.githubusercontent.com/TechwithXalifa/gestukaay/main/transcription/serveur.py
!curl -fsSL -o serveur_synthese.py https://raw.githubusercontent.com/TechwithXalifa/gestukaay/main/synthese/serveur.py
!head -3 serveur_transcription.py serveur_synthese.py

In [ ]:
import subprocess
def vram(titre):
    """Mémoire occupée sur chaque GPU, tous processus confondus (nvidia-smi absent de l'image Kaggle)."""
    code = ("import torch\n"
            "for i in range(torch.cuda.device_count()):\n"
            "    libre, total = torch.cuda.mem_get_info(i)\n"
            "    print(f'GPU {i} {torch.cuda.get_device_name(i)} : {(total - libre) / 2**30:.1f} Go occupés sur {total / 2**30:.1f} Go')")
    print(f"--- VRAM, {titre}")
    print(subprocess.run(["/tmp/venv_tts/bin/python", "-c", code], capture_output=True, text=True).stdout)
import time, requests
def journal(nom, n=4000):
    return open(f"{nom}.log").read()[-n:]
fichiers = {n: open(f"{n}.log", "w") for n in ("transcription", "synthese")}
procs = {
    "transcription": subprocess.Popen(["/tmp/venv_asr/bin/python", "serveur_transcription.py", "--hote", "0.0.0.0", "--port", "8100"],
                                      stdout=fichiers["transcription"], stderr=subprocess.STDOUT, env=os.environ),
    "synthese": subprocess.Popen(["/tmp/venv_tts/bin/python", "serveur_synthese.py", "--hote", "0.0.0.0", "--port", "8200"],
                                 stdout=fichiers["synthese"], stderr=subprocess.STDOUT, env=os.environ),
}
t0 = time.time()
prets = set()
while len(prets) < 2:
    for nom, port in (("transcription", 8100), ("synthese", 8200)):
        if nom in prets:
            continue
        if procs[nom].poll() is not None:  # arrêté : on montre pourquoi, et « Run all » s'arrête ici
            raise RuntimeError(f"{nom} s'est arrêté (code {procs[nom].returncode}). Journal :\n{journal(nom)}")
        try:
            if requests.get(f"http://localhost:{port}/ping", timeout=2).status_code == 200:
                prets.add(nom)
                print(f"{nom} prêt après {time.time() - t0:.0f} s (téléchargement compris)")
        except requests.ConnectionError:
            pass
    if time.time() - t0 > 1800:
        raise RuntimeError("plus de 30 min sans être prêts.\n" + "\n".join(f"== {n}\n{journal(n)}" for n in procs))
    time.sleep(5)
vram("les deux modèles chargés")

In [ ]:
# Aller-retour : une réponse en wolof dite par Oolel, puis retranscrite par M-Kiriku
for nom, p in procs.items():
    if p.poll() is not None:
        raise RuntimeError(f"{nom} s'est arrêté. Journal :\n{journal(nom)}")
H = {"Authorization": "Bearer essai-kaggle"}
phrases = [
    "Am na ñaari milioŋ, ñeenti teemeeri junni ak juróom-benn fukki junni nit ci diiwaanu Cees ci atum ñaari junni ak ñaar-fukk ak ñett.",
    "Njëgu ceeb bi ci Cees, mi ngi tollu ci ñetti teemeer ak juróom-ñeent fukk ak benn ci kilo bi.",
]
for i, phrase in enumerate(phrases, 1):
    t0 = time.time()
    wav = requests.post("http://localhost:8200/v1/audio/speech", headers=H, json={"input": phrase, "response_format": "wav"}, timeout=300).content
    t_synth = time.time() - t0
    import io, wave
    with wave.open(io.BytesIO(wav)) as w:
        duree = w.getnframes() / w.getframerate()
    open(f"reponse_{i}.wav", "wb").write(wav)
    t0 = time.time()
    texte = requests.post("http://localhost:8100/v1/audio/transcriptions", headers=H,
                          files={"file": (f"reponse_{i}.wav", wav, "audio/wav")}, data={"response_format": "json"}, timeout=300).json()["text"]
    t_asr = time.time() - t0
    print(f"Phrase {i} : synthèse {t_synth:.1f} s pour {duree:.1f} s d'audio ; transcription {t_asr:.1f} s")
    print(f"   dit      : {phrase}\n   entendu  : {texte}")
vram("après les deux synthèses et transcriptions")

In [ ]:
from IPython.display import Audio, display
display(Audio("reponse_1.wav"))

In [ ]:
# Journaux des deux serveurs (temps de calcul de chaque appel)
for n in ("transcription", "synthese"):
    print(f"===== {n}"); print(open(f"{n}.log").read()[-2500:])